## The code below is an example created by 경돈.

#### It contains:
#### 1. Basic Bayesian variable selection
#### 2. Labeled Bayesian variable selection
#### 3. Unstructured variable selection that runs both (1) and (2)

Basic Bayesian variable selection

In [1]:
# %% CELL 1 — Project root & sys.path
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_DIR = PROJECT_ROOT / "src"
sys.path.insert(0, str(SRC_DIR))

print("PROJECT_ROOT:", PROJECT_ROOT)
print("SRC_DIR:", SRC_DIR)

PROJECT_ROOT: /Users/jinseunghwan/Documents/development/Data Processing/group-variable-selection
SRC_DIR: /Users/jinseunghwan/Documents/development/Data Processing/group-variable-selection/src


In [2]:
# %% CELL 2 — Imports
import pandas as pd
import numpy as np

from BayesianVariableSelection.bvs import BayesianVariableSelection

In [3]:
# %% CELL 3 — Paths
X_PATH = PROJECT_ROOT / "data" / "examples" / "X.csv"
Y_PATH = PROJECT_ROOT / "data" / "examples" / "Y.csv"

print("X_PATH:", X_PATH)
print("Y_PATH:", Y_PATH)

X_PATH: /Users/jinseunghwan/Documents/development/Data Processing/group-variable-selection/data/examples/X.csv
Y_PATH: /Users/jinseunghwan/Documents/development/Data Processing/group-variable-selection/data/examples/Y.csv


In [4]:
# %% CELL 4 — Load X and y
X = pd.read_csv(X_PATH)
y_df = pd.read_csv(Y_PATH)

# If Y.csv has a single column, squeeze into a Series
if y_df.shape[1] == 1:
    y = y_df.iloc[:, 0]
else:
    # If it has multiple columns, you must choose one
    # (Keeping deterministic default: first column)
    y = y_df.iloc[:, 0]

# Basic checks
print("X shape:", X.shape)
print("y shape:", y.shape)
print("y head:")
display(y.head())

# Ensure numeric y
y = pd.to_numeric(y, errors="coerce")
print("y numeric describe:")
print(y.describe())

# Drop rows with missing y (and align X)
mask = ~y.isna()
X = X.loc[mask].reset_index(drop=True)
y = y.loc[mask].reset_index(drop=True)

print("After dropping NA y:")
print("X shape:", X.shape)
print("y shape:", y.shape)


X shape: (1000, 100)
y shape: (1000,)
y head:


0    69.702397
1    59.777245
2    59.765703
3    61.937234
4    54.124244
Name: x, dtype: float64

y numeric describe:
count    1000.000000
mean       45.648723
std        14.945972
min         1.890083
25%        35.137402
50%        45.887801
75%        56.428813
max        95.588017
Name: x, dtype: float64
After dropping NA y:
X shape: (1000, 100)
y shape: (1000,)


In [45]:
selector = BayesianVariableSelection(
    X.fillna(0).to_numpy(),
    y.to_numpy().reshape(-1),
    track = True,
)

In [48]:
# Model Initialize
selector.initialize(segment_no = 3, iteration = 1000, burnin = 2000, cutoff = 0.03, step = 1)
selector.fit()

  0%|          | 0/3000 [00:00<?, ?it/s]

In [ ]:
selector.visualize_topic_seg(0)

Labeled Bayesian variable selection

In [10]:
# Model Setup
from BayesianVariableSelection import LabelBVS

selector = LabelBVS(
    X.fillna(0), # X: word frequency matrix
    y, # y: rating 
    dist_type = 'cooccur',
    track = True,
)

In [13]:
# Model Initialize
selector.initialize(segment_no = 3, iteration = 100, burnin = 2000, cutoff = 0.03, step = 1)
selector.fit()

  0%|          | 0/2100 [00:00<?, ?it/s]

In [ ]:
selector.visualize_topic_seg(0)

Unstructured variable selection that runs both (1) and (2)

In [26]:
data_path = PROJECT_ROOT / "data" / "RAW" / "251205_ParisHotel_comment_score.csv"

In [27]:
data = pd.read_csv(data_path)

In [28]:
data['content'] = data['title'].fillna('').str.cat(data['text'], sep=' ')

In [29]:
# Review-level data
data[['id', 'locationId', 'content', 'rating']]

,id,locationId,content,rating
0,1009363551,23688737.0,Wonderful 5-star experience in the 1st. Perfe...,5.0
1,1041040518,23688737.0,Madame Reve ALL THE WAY!!! We decided to go to...,5.0
2,1040950353,23688737.0,Overpriced My partner and I stayed in a Deluxe...,3.0
3,1040312147,23688737.0,Shocking breach of privacy at a so-called five...,2.0
4,1040284170,23688737.0,Paris Good location !\nThe service in the room...,4.0
...,...,...,...,...
103880,834074624,617625.0,Great Boutique style hotel Great location clos...,5.0
103881,833748659,617625.0,Amazing property and amazing room We were so h...,5.0
103882,833729338,617625.0,We love this hotel. Me and my daughter have st...,5.0
103883,833478344,617625.0,Fantastic 3 nights at Grand Hotel du Palais Ro...,5.0


In [30]:
data['locationId_month'] = data['locationId'].astype(str) + "_" + data['publishedDate'].astype('datetime64[ns]').dt.to_period('M').astype(str)

In [31]:
from BayesianVariableSelection.unstructured import UnstructureVariableSelection

selector = UnstructureVariableSelection(
    data = data,
    review_col = 'content',
    rating_col = 'rating',
    aggregate_col = 'locationId',
    word_corpus_file = None,
    dist_type = 'correlate', # Distance Type: correlate (Correlation; aggregate) / cooccur (Cooccurrence; review-level)
    track = True
)

In [32]:
_ = selector.generate(top_n = 300)

300 of words are selected


In [33]:
corpus_export = PROJECT_ROOT / "output" / "example_word_corpus.xlsx"
selector.export_word_corpus(corpus_export)

In [ ]:
# Check the exported word corpus file and modify it as needed before re-importing
# exported corpus file must have two columns; put lemma next to 'corpus' column
# before running below code

selector = UnstructureVariableSelection(
    data = data,
    review_col = 'content',
    rating_col = 'rating',
    aggregate_col = 'locationId',
    word_corpus_file = corpus_export,
    dist_type = 'correlate', # Distance Type: correlate (Correlation; aggregate) / cooccur (Cooccurrence; review-level)
    track = True
)

In [36]:
_ = selector.generate()

300 of words are selected


In [37]:
# Correlate: cutoff = 0.7 / Cooccur: cutoff = 0.3
selector.initialize(segment_no = 3, iteration = 300, burnin = 300, cutoff = 0.7, step = 1)
selector.fit()

  0%|          | 0/600 [00:00<?, ?it/s]

/Users/jinseunghwan/Documents/development/Data Processing/group-variable-selection/.venv/lib/python3.11/site-packages/numpy/lib/_function_base_impl.py:3023: RuntimeWarning:

invalid value encountered in divide

/Users/jinseunghwan/Documents/development/Data Processing/group-variable-selection/.venv/lib/python3.11/site-packages/numpy/lib/_function_base_impl.py:3024: RuntimeWarning:

invalid value encountered in divide



In [38]:
selector.update(500)

  0%|          | 0/500 [00:00<?, ?it/s]

/Users/jinseunghwan/Documents/development/Data Processing/group-variable-selection/.venv/lib/python3.11/site-packages/numpy/lib/_function_base_impl.py:3015: RuntimeWarning:

Degrees of freedom <= 0 for slice

/Users/jinseunghwan/Documents/development/Data Processing/group-variable-selection/.venv/lib/python3.11/site-packages/numpy/lib/_function_base_impl.py:2888: RuntimeWarning:

divide by zero encountered in divide

/Users/jinseunghwan/Documents/development/Data Processing/group-variable-selection/.venv/lib/python3.11/site-packages/numpy/lib/_function_base_impl.py:2888: RuntimeWarning:

invalid value encountered in multiply

/Users/jinseunghwan/Documents/development/Data Processing/group-variable-selection/.venv/lib/python3.11/site-packages/numpy/lib/_function_base_impl.py:576: RuntimeWarning:

Mean of empty slice

/Users/jinseunghwan/Documents/development/Data Processing/group-variable-selection/.venv/lib/python3.11/site-packages/numpy/_core/_methods.py:134: RuntimeWarning:

invalid 

In [39]:
selector.visualize_topic_seg(2, k = 3, cut = 0.9).update_layout(width=1200, height=600, xaxis_tickangle = 45)

In [40]:
np.unique(selector.find_membership(), return_counts=True)

(array([0, 1, 2]), array([299, 538,  67]))

In [41]:
selector.find_appropriate_document('appreciate')

,locationId,rating,content
0,23688737.0,5.0,Wonderful 5-star experience in the 1st. Perfe...
1,23688737.0,5.0,Madame Reve ALL THE WAY!!! We decided to go to...
2,23688737.0,3.0,Overpriced My partner and I stayed in a Deluxe...
3,23688737.0,2.0,Shocking breach of privacy at a so-called five...
4,23688737.0,4.0,Paris Good location !\nThe service in the room...
...,...,...,...
103880,617625.0,5.0,Great Boutique style hotel Great location clos...
103881,617625.0,5.0,Amazing property and amazing room We were so h...
103882,617625.0,5.0,We love this hotel. Me and my daughter have st...
103883,617625.0,5.0,Fantastic 3 nights at Grand Hotel du Palais Ro...


In [42]:
selector.diagnose_model()

,variable,parameter,std_err,t_stat,p_value,CI_low,CI_upp
9,absolutely,-0.4797,0.208,-2.302,0.022,-0.889,-0.070
14,although,0.5895,0.242,2.435,0.015,0.114,1.065
15,always,-0.3960,0.153,-2.580,0.010,-0.697,-0.095
17,amazing,0.9320,0.303,3.079,0.002,0.338,1.526
23,arrival,-0.9148,0.249,-3.674,0.000,-1.404,-0.426
...,...,...,...,...,...,...,...
285,wait,0.4795,0.229,2.097,0.036,0.030,0.929
293,welcome,0.3500,0.114,3.080,0.002,0.127,0.573
294,well,0.1890,0.090,2.103,0.036,0.013,0.365
299,work,-0.3617,0.126,-2.875,0.004,-0.609,-0.115


In [43]:
selector.find_correlation_by_seg(0)

,index,select_ratio,coef|z=1,coef
corpus,,,,


In [44]:
freq = selector.frequency_matrix.copy()
freq['membership'] = selector.find_membership()